# 06 Noise

05 with noise. Ambient-noise correlations never converge fully: each station pair's correlation carries noise
that shrinks the longer the pair records together. Here every pair gets correlation noise with standard deviation

    NSR x SIGNAL x sqrt(DAYS_REF / days)

where `days` is how long the pair recorded together and `SIGNAL` is the rms echo of the `REFERENCE` design
(noise-free, from 05's forward model). So `NSR = 1` means the noise equals the echo after `DAYS_REF` days of shared
recording. The real level at Rotokawa is unknown, so `NSR_LIST` scans it.

How time enters:

- Trilliums, Mercury and GeoNet record all deployment, so their pairs are the cleanest.
- Static nodes record all deployment too; phased nodes only during their phase, so their pairs are noisier.
- The phase 1 pick uses only the data recorded **by the end of phase 1** (permanent stations included).
- Each pair has one fixed noise record (a draw per week, seeded by the pair and `REAL`). The phase 1 pick and the
  final image see the same noise on the data they share, and designs that share stations see the same noise, so
  differences between designs come from the designs.

Same forward model, imaging, picks and scores as 05 (`mi_common.py`).

Inputs: everything 05 reads, and 05's images for comparison. Outputs: `data/results_06_noise.csv`, images in
`data/mi_images_06/`

In [ ]:
# --- Imports ---
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mi_common import MatrixImaging

In [ ]:
# --- Settings ---
SCENARIOS = {"dome": (0.0, 0.0)}      # as in 05
DESIGNS = None                         # None = every design in designs.json
NSR_LIST = [1.0]                       # noise / signal after DAYS_REF days shared. Scan e.g. [0.5, 1.0, 2.0]
DAYS_REF = 30
REFERENCE = "static: wide grid"        # design whose rms echo defines SIGNAL (fixed, so levels compare across designs)
REAL = 0                               # noise realisation
OUT = "../data/mi_images_06"
RESULTS = "../data/results_06_noise.csv"
OUT_05 = "../data/mi_images_05"        # noise-free images from 05, shown alongside

mi = MatrixImaging(scenarios=SCENARIOS)
DESIGNS = DESIGNS or mi.designs()
os.makedirs(OUT, exist_ok=True)
SIGNAL = {name: mi.signal(name, REFERENCE) for name in SCENARIOS}
print(mi)
print("SIGNAL (rms echo, " + REFERENCE + "):", {k: f"{v:.2e}" for k, v in SIGNAL.items()})

## Run every design at every noise level

Saved images are reused on a rerun (delete `data/mi_images_06/` after changing 02, 03 or the settings here).

In [ ]:
# --- Image every design for every scenario and noise level ---
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
fname = lambda name, design, nsr: f"{OUT}/{slug(name)}__{slug(design)}__nsr{nsr:g}_r{REAL}.npz"
rows = []
for nsr in NSR_LIST:
    for name in SCENARIOS:
        noise = mi.noise_model(nsr * SIGNAL[name], days_ref=DAYS_REF, real=REAL)
        for design in DESIGNS:
            f = fname(name, design, nsr)
            r = np.load(f, allow_pickle=True) if os.path.exists(f) else None
            if r is None or r["image"].shape != (len(mi.IX), len(mi.IY), len(mi.IZ)):
                out = mi.run_design(design, name, noise=noise)
                nodes = [(s, q) for q, p in enumerate(out["phases"]) for s in p["stations"]]
                info = dict(stations=len(out["ids"]), pairs=out["n_pairs"], seconds=round(out["seconds"]))
                np.savez_compressed(f, image=out["image"], phase1=out.get("phase1", np.zeros(0)),
                                    target=out.get("target", np.full(2, np.nan)), node_ids=[s for s, _ in nodes],
                                    node_dep=[q for _, q in nodes], info=np.array(info, dtype=object))
                r = np.load(f, allow_pickle=True)
                print(f"NSR {nsr:g} / {name} / {design}: {out['seconds']:.0f} s", flush=True)
            rows.append(dict(nsr=nsr, scenario=name, design=design, **r["info"].item(), **mi.score(r["image"], name)))
res = pd.DataFrame(rows)
res.to_csv(RESULTS, index=False)

In [ ]:
# --- Scores with noise, next to 05's noise-free ones ---
f05 = "../data/results_05_mi.csv"
cols = ["area_coverage", "crest_imaged_km", "target_error_km"]
t = res.pivot_table(index=["scenario", "design"], columns="nsr", values=cols).round(2)
t.columns = [f"{c} (NSR {n:g})" for c, n in t.columns]
if os.path.exists(f05):
    r05 = pd.read_csv(f05).set_index(["scenario", "design"])[cols].round(2)
    t = r05.add_suffix(" (no noise)").join(t)
order = lambda c: (cols.index(c.split(" (")[0]), "no noise" not in c, float(c.split("NSR ")[1][:-1]) if "NSR" in c else 0)
t[sorted(t.columns, key=order)]

## Recovered images

One row per design: 05's noise-free image, then the image at each noise level. Sections along the node axis
through the BDT high, at true scale; the hatched band is the true BDT zone; each image is normalised to its own
maximum.

In [ ]:
# --- Plot helpers (as 05) ---
ST = mi.stations
STY = dict(
    rt=dict(marker="^", mfc="#ea6a2c", mec="#3d2414", mew=0.8, ms=5, ls="none"),
    ns=dict(marker="v", mfc="#1f9e78", mec="#14332a", mew=0.8, ms=5, ls="none"),
    gn=dict(marker="D", mfc="#6a3d9a", mec="#2b1640", mew=0.8, ms=4, ls="none"),
    tr=dict(marker="s", mfc="none", mec="#2a78d6", mew=1.4, ms=5, ls="none"),
    drill=dict(marker="*", mfc="#f5c518", mec="#0b0b0b", mew=0.8, ms=12, ls="none"),
    n1=dict(marker="o", mfc="white", mec="#0b0b0b", mew=1.4, ms=5.5, ls="none"),
    n2=dict(marker="o", mfc="white", mec="#0b0b0b", mew=0.7, ms=3.5, ls="none"),
)
GROUPS = {"rt": (ST.type == "mercury") & ST.name.fillna("").str.startswith("RT"),
          "ns": (ST.type == "mercury") & ST.name.fillna("").str.startswith("NS"),
          "gn": ST.type == "geonet", "tr": ST.type == "trillium"}
PATCH_COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
th = np.linspace(0, 2 * np.pi, 200)
R_MAP = mi.IX[-1]
plt.rcParams["hatch.linewidth"] = 0.8


def bdt_zone(ax, scenario, alpha=0.45):
    s, _, _, top, base = mi.velocity_section(scenario)
    ax.fill_between(s, top, base, facecolor="none", edgecolor=(1, 1, 1, alpha), hatch="///", lw=0)
    ax.plot(s, top, "--", color="white", lw=1.5)
    ax.plot(s, base, ":", color="white", lw=1.5)


def stations_map(ax):
    ax.plot(mi.MT_R * np.cos(th), mi.MT_R * np.sin(th), "--", color="#3d3c39", lw=1.1)
    for k, sel in GROUPS.items():
        ax.plot(ST.x[sel], ST.y[sel], **STY[k])
    ax.plot(0, 0, **STY["drill"])


def nodes_map(ax, r, design):
    st, dep = r["node_ids"], r["node_dep"]
    if mi.is_phased(design):
        ax.plot(ST.x[st[dep == 0]], ST.y[st[dep == 0]], **STY["n1"])
        for q in range(1, dep.max() + 1):
            ax.plot(ST.x[st[dep == q]], ST.y[st[dep == q]], "o", ms=3.2, mfc=PATCH_COL[(q - 1) % 6], mec="#0b0b0b",
                    mew=0.4)
    else:
        ax.plot(ST.x[st], ST.y[st], **STY["n2"])


def images(name, design):
    # [(label, saved file)]: 05's noise-free image if present, then each noise level
    out = []
    f05 = f"{OUT_05}/{slug(name)}__{slug(design)}.npz"
    if os.path.exists(f05):
        out.append(("no noise", np.load(f05, allow_pickle=True)))
    out += [(f"NSR {nsr:g}", np.load(fname(name, design, nsr), allow_pickle=True)) for nsr in NSR_LIST]
    return out

In [ ]:
# --- Sections: one row per design, noise-free then each noise level ---
for name in SCENARIOS:
    nc = len(images(name, DESIGNS[0]))
    fig, axs = plt.subplots(len(DESIGNS), nc, figsize=(5.2 * nc + 1, 2.9 * len(DESIGNS)), sharex=True, sharey=True,
                            squeeze=False, layout="constrained")
    for row, design in zip(axs, DESIGNS):
        for ax, (lab, r) in zip(row, images(name, design)):
            s, z, sec = mi.section(r["image"], name)
            pm = ax.pcolormesh(s, z, sec.T, cmap="magma", vmin=0, vmax=1, shading="auto", rasterized=True)
            bdt_zone(ax, name)
            cov = mi.score(r["image"], name)["area_coverage"]
            ax.set(ylim=(mi.IZ[-1], 0), aspect=1)
            ax.set_title(f"{design}, {lab}   ({cov:.2f})", loc="left", fontsize=9.5)
    for ax in axs[:, 0]:
        ax.set_ylabel("Depth (km)")
    for ax in axs[-1]:
        ax.set_xlabel("Distance along the node axis (km)")
    fig.colorbar(pm, ax=axs, label="Image amplitude (normalised)", shrink=0.4)
    fig.suptitle(f"{name}: recovered images (in brackets: MT-area coverage)", fontsize=12)
    plt.show()

In [ ]:
# --- Map view at each noise level: picked BDT top minus true (white = right, red = too shallow, grey = not imaged) ---
for nsr in NSR_LIST:
    for name in SCENARIOS:
        n, nc = len(DESIGNS), 3
        fig, axs = plt.subplots(int(np.ceil(n / nc)), nc, figsize=(15, 5 * np.ceil(n / nc)), sharex=True, sharey=True,
                                squeeze=False, layout="constrained")
        for ax, design in zip(axs.ravel(), DESIGNS):
            r = np.load(fname(name, design, nsr), allow_pickle=True)
            err = mi.pick_top(r["image"]) - mi.true_top(name)
            ax.set_facecolor("#d9d8d3")
            pc = ax.pcolormesh(mi.IX, mi.IY, err.T, cmap="RdBu", vmin=-1, vmax=1, shading="auto", rasterized=True)
            stations_map(ax)
            nodes_map(ax, r, design)
            ax.set(aspect="equal", xlim=(-R_MAP, R_MAP), ylim=(-R_MAP, R_MAP))
            ax.set_title(f"{design}   ({mi.score(r['image'], name)['area_coverage']:.2f})", loc="left", fontsize=10)
        for ax in axs.ravel()[n:]:
            ax.axis("off")
        fig.colorbar(pc, ax=axs, label="Picked minus true BDT top (km)", shrink=0.5)
        fig.suptitle(f"{name}, NSR {nsr:g}: where each design recovers the BDT top (dashed: MT target area)",
                     fontsize=12)
        plt.show()

## Phased designs: the phase 1 pick under noise

The pick is made from the data recorded by the end of phase 1 only. If it misses, phase 2 goes to the wrong place.

In [ ]:
# --- Phase 1 pick at each noise level ---
phased = [d for d in DESIGNS if mi.is_phased(d)]
for name in SCENARIOS:
    c = mi.centre(name)
    for design in phased:
        fig, axs = plt.subplots(1, len(NSR_LIST), figsize=(5 * len(NSR_LIST), 4.8), squeeze=False,
                                layout="constrained")
        for ax, nsr in zip(axs[0], NSR_LIST):
            r = np.load(fname(name, design, nsr), allow_pickle=True)
            top1 = mi.pick_top(r["phase1"])
            ax.set_facecolor("#d9d8d3")
            ax.pcolormesh(mi.IX, mi.IY, (top1 - mi.true_top(name)).T, cmap="RdBu", vmin=-1, vmax=1, shading="auto")
            stations_map(ax)
            nodes_map(ax, r, design)
            ax.plot(*r["target"], "+", color="#e34948", ms=16, mew=3)
            ax.plot(*c, "x", color="#0b0b0b", ms=12, mew=2.5)
            ax.set(aspect="equal", xlim=(-R_MAP, R_MAP), ylim=(-R_MAP, R_MAP),
                   title=f"NSR {nsr:g}: phase 1 pick {np.hypot(*(r['target'] - c)):.2f} km off")
        fig.suptitle(f"{name}: {design} (red +: pick from phase 1; black x: true BDT high)", fontsize=11)
        plt.show()